<a href="https://colab.research.google.com/github/angeruzzi/recommender_system_movielens/blob/main/02_evaluation_framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluation Framework — MovieLens 100K

## 1. Introdução

Este notebook define o protocolo experimental que será utilizado para avaliar e comparar os diferentes sistemas de recomendação desenvolvidos neste projeto.

Em vez de avaliar cada modelo utilizando procedimentos específicos, será construído um framework comum de avaliação. Dessa forma, modelos baseados em popularidade, conteúdo, Collaborative Filtering e fatores latentes poderão ser comparados sob as mesmas condições.

Como o objetivo principal do projeto é gerar listas personalizadas de recomendações, a avaliação será orientada para **Top-K Recommendation**.

O protocolo será responsável por definir:

- a separação entre dados de treino e teste;
- o que será considerado um item relevante;
- quais itens poderão ser candidatos à recomendação;
- como serão geradas as listas Top-K;
- as métricas utilizadas para avaliar essas listas.

A dimensão temporal das interações será preservada durante a separação dos dados, utilizando informações passadas para avaliar a capacidade dos modelos de recuperar itens relevantes em interações futuras.

## 2. Preparação dos Dados

O framework utilizará as avaliações do MovieLens 100K.

Como as análises exploratórias já foram realizadas anteriormente, neste notebook serão carregadas apenas as informações necessárias para a construção e avaliação dos modelos.

In [1]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -q -o ml-100k.zip

In [3]:
import numpy as np
import pandas as pd

In [6]:
ratings = pd.read_csv(
    "/content/ml-100k/u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)

ratings["datetime"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s"
)

ratings.head()

,user_id,item_id,rating,timestamp,datetime
0,196,242,3,881250949,1997-12-04 15:55:49
1,186,302,3,891717742,1998-04-04 19:22:22
2,22,377,1,878887116,1997-11-07 07:18:36
3,244,51,2,880606923,1997-11-27 05:02:03
4,166,346,1,886397596,1998-02-02 05:33:16


In [4]:
genres = [
    "unknown", "Action", "Adventure", "Animation",
    "Children", "Comedy", "Crime", "Documentary",
    "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi",
    "Thriller", "War", "Western"
]

movie_columns = [
    "item_id",
    "title",
    "release_date",
    "video_release_date",
    "imdb_url"
] + genres

movies = pd.read_csv(
    "/content/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    names=movie_columns
)

,item_id,title,release_date,video_release_date,imdb_url,unknown,Action,Adventure,Animation,Children,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


### 2.2 Validação dos Dados

Antes da construção do protocolo experimental, são realizadas algumas verificações básicas para garantir que os dados carregados correspondem ao conjunto esperado.

In [7]:
print(f"Avaliações: {len(ratings):,}")
print(f"Usuários:   {ratings['user_id'].nunique():,}")
print(f"Filmes:     {ratings['item_id'].nunique():,}")

print(
    f"Período:    {ratings['datetime'].min()} "
    f"até {ratings['datetime'].max()}"
)

print(
    f"Ratings:    {sorted(ratings['rating'].unique())}"
)

Avaliações: 100,000
Usuários:   943
Filmes:     1,682
Período:    1997-09-20 03:05:10 até 1998-04-22 23:10:38
Ratings:    [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


### 2.3 Configuração Experimental

Os principais parâmetros utilizados pelo protocolo experimental são centralizados nesta seção para garantir consistência e facilitar a reprodução dos experimentos.

O limiar de relevância define quais avaliações serão consideradas sinais positivos de preferência. A proporção de treino determina a divisão temporal das interações de cada usuário e \(K\) representa o tamanho padrão das listas de recomendação.

In [39]:
RELEVANCE_THRESHOLD = 4
TRAIN_RATIO = 0.8
K = 10

print(f"Relevance threshold: {RELEVANCE_THRESHOLD}")
print(f"Train ratio:         {TRAIN_RATIO:.0%}")
print(f"Top-K:               {K}")

Relevance threshold: 4
Train ratio:         80%
Top-K:               10


## 3. Definição do Problema de Recomendação

### 3.1 Feedback Explícito e Relevância

O MovieLens 100K fornece feedback explícito na forma de avaliações de 1 a 5 estrelas.

Entretanto, em um problema de Top-K Recommendation, o objetivo não é apenas prever o valor exato de um rating. O sistema deve ordenar itens de modo que os mais relevantes para cada usuário apareçam nas primeiras posições da lista de recomendações.

Para este projeto, uma interação será considerada **relevante** quando o usuário tiver atribuído rating igual ou superior a 4:

\[
relevant(u,i)=
\begin{cases}
1, & \text{se } rating_{ui} \ge 4 \\
0, & \text{caso contrário}
\end{cases}
\]

Assim, ratings 4 e 5 serão tratados como sinais positivos de preferência, enquanto ratings 1, 2 e 3 não serão considerados itens relevantes para as métricas Top-K.

### 3.2 Top-K Recommendation

Para cada usuário \(u\), o sistema de recomendação atribuirá um score aos itens candidatos e produzirá uma lista ordenada contendo os \(K\) itens com maior score.

Essa lista pode ser representada por:

\[
Rec_K(u) = [i_1, i_2, \ldots, i_K]
\]

onde os itens estão ordenados do maior para o menor score estimado pelo modelo.

O objetivo da avaliação será verificar em que medida os itens relevantes observados no período de teste aparecem nas primeiras posições dessa lista.

In [8]:
K = 10

### 3.3 Ground Truth

O **ground truth** representa o conjunto de itens que serão considerados relevantes para um usuário durante a avaliação.

Como o protocolo será temporal, o modelo será treinado utilizando interações anteriores e avaliado utilizando interações posteriores.

Para cada usuário \(u\), o conjunto de itens relevantes do período de teste será definido como:

\[
GT(u) =
\{i \in Test(u) \mid rating_{ui} \ge 4\}
\]

O sistema será avaliado verificando se os itens presentes em \(GT(u)\) aparecem entre as recomendações Top-K produzidas pelo modelo.

### 3.4 Interações Não Relevantes no Teste

Uma interação presente no conjunto de teste não é automaticamente considerada uma recomendação correta.

Por exemplo, um filme avaliado com rating 1 ou 2 indica que o usuário interagiu com o item, mas não representa necessariamente uma preferência que o sistema deveria tentar recomendar.

Por esse motivo, apenas as interações que atingirem o limiar de relevância serão utilizadas como ground truth positivo nas métricas de ranking.

### 3.5 Usuários Elegíveis para Avaliação

As métricas de ranking serão calculadas apenas para usuários que possuam pelo menos um item relevante no conjunto de teste.

Usuários sem interações relevantes no período de teste não possuem um ground truth positivo contra o qual as recomendações possam ser comparadas.

Essa regra evita atribuir artificialmente desempenho positivo ou negativo a casos nos quais não existe item relevante a ser recuperado.

### 3.6 Universo de Itens Candidatos

Para cada usuário, o sistema deverá produzir recomendações entre itens elegíveis do catálogo.

Itens já observados no histórico de treino do usuário serão excluídos das recomendações, pois o objetivo é sugerir itens que ainda não fazem parte de seu histórico conhecido.

Inicialmente, o conjunto de candidatos será definido como:

\[
Candidates(u) =
Catalog_{train} - Seen_{train}(u)
\]

onde \(Catalog_{train}\) representa os itens conhecidos durante o treinamento e \(Seen_{train}(u)\) representa os itens já avaliados pelo usuário no conjunto de treino.

### 3.7 Convenções Utilizadas no Projeto

| Elemento | Definição |
|---|---|
| Feedback | Rating explícito de 1 a 5 |
| Item relevante | Rating ≥ 4 |
| K padrão | 10 |
| Ground Truth | Itens relevantes presentes no teste |
| Usuário elegível | Possui pelo menos 1 item relevante no teste |
| Itens vistos | Itens presentes no histórico de treino |
| Candidate Items | Itens conhecidos no treino e ainda não vistos pelo usuário |
| Objetivo | Maximizar a qualidade do ranking Top-K |

## 4. Estratégia de Separação dos Dados

### 4.1 Split Temporal por Usuário

Para aproximar a avaliação offline de um cenário real de recomendação, as interações serão separadas respeitando a ordem temporal de cada usuário.

Para cada usuário, as interações serão ordenadas pelo timestamp e divididas em:

- **treino**: interações mais antigas;
- **teste**: interações mais recentes.

Neste projeto será utilizada inicialmente uma proporção de 80% para treino e 20% para teste.

In [9]:
TRAIN_RATIO = 0.8

In [10]:
def temporal_split_by_user(ratings, train_ratio=0.8):
    train_parts = []
    test_parts = []

    for user_id, user_data in ratings.groupby("user_id"):
        user_data = user_data.sort_values("datetime")

        split_idx = int(len(user_data) * train_ratio)

        train_parts.append(user_data.iloc[:split_idx])
        test_parts.append(user_data.iloc[split_idx:])

    train = pd.concat(train_parts).reset_index(drop=True)
    test = pd.concat(test_parts).reset_index(drop=True)

    return train, test

In [11]:
train, test = temporal_split_by_user(
    ratings,
    train_ratio=TRAIN_RATIO
)

print(f"Train: {len(train):,}")
print(f"Test:  {len(test):,}")

Train: 79,619
Test:  20,381


In [13]:
len(train) + len(test) == len(ratings)

True

In [14]:
assert len(train) + len(test) == len(ratings)

### 4.2 Distribuição dos Conjuntos

Após a separação temporal, é importante verificar a quantidade de interações em cada conjunto e garantir que todos os usuários estejam adequadamente representados.

In [15]:
print(
    f"Train: {len(train):,} "
    f"({len(train) / len(ratings):.2%})"
)

print(
    f"Test:  {len(test):,} "
    f"({len(test) / len(ratings):.2%})"
)

print()
print(f"Usuários no train: {train['user_id'].nunique()}")
print(f"Usuários no test:  {test['user_id'].nunique()}")

Train: 79,619 (79.62%)
Test:  20,381 (20.38%)

Usuários no train: 943
Usuários no test:  943


### 4.3 Validação da Ordem Temporal

A validade do protocolo depende de garantir que nenhuma interação futura seja utilizada durante o treinamento para prever uma interação anterior.

Para cada usuário, a última interação presente no conjunto de treino deve ocorrer antes ou no mesmo instante da primeira interação presente no conjunto de teste.

In [16]:
train_last = (
    train
    .groupby("user_id")["datetime"]
    .max()
)

test_first = (
    test
    .groupby("user_id")["datetime"]
    .min()
)

temporal_validation = (
    train_last <= test_first
)

print(
    f"Usuários com split temporal válido: "
    f"{temporal_validation.mean():.2%}"
)

Usuários com split temporal válido: 100.00%


In [17]:
assert temporal_validation.all()

### 4.4 Tamanho do Histórico por Usuário

Vale conferir quantas interações cada usuário ficou em treino e teste.

In [18]:
split_stats = pd.DataFrame({
    "train_interactions": train.groupby("user_id").size(),
    "test_interactions": test.groupby("user_id").size()
})

split_stats.describe()

,train_interactions,test_interactions
count,943.000000,943.000000
mean,84.431601,21.612937
std,80.746499,20.187753
min,16.000000,4.000000
25%,26.000000,7.000000
50%,52.000000,13.000000
75%,118.000000,30.000000
max,589.000000,148.000000


### 4.5 Construção do Ground Truth

Após o split, o ground truth de cada usuário será composto apenas pelas interações relevantes observadas no conjunto de teste.

Uma interação é considerada relevante quando possui rating maior ou igual ao limiar definido anteriormente.

In [30]:
RELEVANCE_THRESHOLD = 4

test_relevant = test[
    test["rating"] >= RELEVANCE_THRESHOLD
].copy()

ground_truth = (
    test_relevant
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

### 4.6 Usuários Elegíveis

Agora aplicamos a regra definida anteriormente: só avaliamos usuários com pelo menos um item relevante no teste.

In [31]:
eligible_users = list(ground_truth.keys())

print(f"Usuários totais:    {ratings['user_id'].nunique()}")
print(f"Usuários elegíveis: {len(eligible_users)}")

print(
    f"Percentual elegível: "
    f"{len(eligible_users) / ratings['user_id'].nunique():.2%}"
)

Usuários totais:    943
Usuários elegíveis: 908
Percentual elegível: 96.29%


In [ ]:
relevant_per_user = pd.Series({
    user_id: len(items)
    for user_id, items in ground_truth.items()
})

relevant_per_user.describe()

### 4.7 Catálogo Disponível no Treino

Agora definimos formalmente os itens conhecidos pelo sistema.

In [20]:
train_catalog = set(
    train["item_id"].unique()
)

print(
    f"Itens conhecidos no treino: "
    f"{len(train_catalog):,}"
)

Itens conhecidos no treino: 1,611


In [21]:
full_catalog = set(
    ratings["item_id"].unique()
)

new_items_in_test = (
    set(test["item_id"].unique())
    - train_catalog
)

print(
    f"Itens presentes no teste e ausentes do treino: "
    f"{len(new_items_in_test)}"
)

Itens presentes no teste e ausentes do treino: 71


### 4.8 Candidate Items por Usuário

A função será:

Candidates(u)=Catalog_train − Seen_train(u)

In [22]:
seen_items = (
    train
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

In [23]:
def get_candidate_items(user_id):
    seen = seen_items.get(user_id, set())

    return train_catalog - seen

In [32]:
user_id = eligible_users[0]

candidates = get_candidate_items(user_id)

print(f"Usuário: {user_id}")
print(f"Itens vistos no treino: {len(seen_items[user_id])}")
print(f"Itens candidatos:       {len(candidates)}")

Usuário: 1
Itens vistos no treino: 217
Itens candidatos:       1394


### 4.9 Validação dos Candidate Items

Vamos garantir que nenhum item já visto esteja sendo recomendado como candidato:

In [33]:
user_id = eligible_users[0]

intersection = (
    get_candidate_items(user_id)
    & seen_items[user_id]
)

print(
    f"Itens simultaneamente vistos e candidatos: "
    f"{len(intersection)}"
)

Itens simultaneamente vistos e candidatos: 0


In [34]:
for user_id in eligible_users:
    assert (
        get_candidate_items(user_id)
        .isdisjoint(seen_items[user_id])
    )

### 4.10 Um detalhe importante sobre o ground truth

Aqui existe uma questão que vale verificar:

um item relevante presente no teste pode não estar em train_catalog.

Se isso acontecer, nenhum modelo puramente colaborativo treinado apenas com train conseguirá recomendá-lo.

In [35]:
unreachable_relevant = {}

for user_id, relevant_items in ground_truth.items():
    unreachable = relevant_items - train_catalog

    if unreachable:
        unreachable_relevant[user_id] = unreachable

In [36]:
n_unreachable = sum(
    len(items)
    for items in unreachable_relevant.values()
)

print(
    f"Itens relevantes de teste ausentes do catálogo de treino: "
    f"{n_unreachable}"
)

Itens relevantes de teste ausentes do catálogo de treino: 17


### 4.11 Ground Truth Avaliável

In [37]:
evaluable_ground_truth = {
    user_id: relevant_items & train_catalog
    for user_id, relevant_items in ground_truth.items()
}

In [38]:
evaluable_ground_truth = {
    user_id: items
    for user_id, items in evaluable_ground_truth.items()
    if len(items) > 0
}

evaluation_users = list(
    evaluable_ground_truth.keys()
)

print(
    f"Usuários disponíveis para avaliação: "
    f"{len(evaluation_users)}"
)

Usuários disponíveis para avaliação: 907


### 4.12 Principais Observações

O conjunto de dados foi separado temporalmente por usuário, preservando aproximadamente 80% das interações mais antigas para treinamento e 20% das interações mais recentes para teste.

As verificações realizadas garantem que:

- as interações de treino precedem temporalmente as interações de teste de cada usuário;
- itens já observados no histórico de treino são removidos do universo de candidatos;
- o ground truth considera apenas avaliações relevantes no período de teste;
- usuários sem itens relevantes no teste são excluídos das métricas Top-K;
- itens relevantes ausentes do catálogo de treino são identificados como casos de item cold start;
- o conjunto de avaliação pode ser restringido a itens efetivamente alcançáveis pelos modelos.

Com esse protocolo, todos os modelos posteriores poderão ser avaliados sob as mesmas condições.

## 5. Métricas Top-K

### 5.1 Precision@K

A Precision@K mede qual proporção dos itens recomendados entre as primeiras \(K\) posições é relevante para o usuário.

\[
Precision@K =
\frac{|Rec_K(u) \cap GT(u)|}{K}
\]

Valores maiores indicam que uma parcela maior da lista recomendada contém itens relevantes.

In [40]:
def precision_at_k(recommended_items, relevant_items, k=10):
    recommended_k = recommended_items[:k]

    if k == 0:
        return 0.0

    hits = len(set(recommended_k) & set(relevant_items))

    return hits / k

In [41]:
#exemplo

recommended = [10, 20, 30, 40, 50]
relevant = {20, 40, 60}

precision_at_k(
    recommended_items=recommended,
    relevant_items=relevant,
    k=5
)

0.4

### 5.2 Recall@K

A Recall@K mede qual proporção dos itens relevantes existentes no ground truth foi recuperada pelo sistema entre as primeiras \(K\) recomendações.

\[
Recall@K =
\frac{|Rec_K(u) \cap GT(u)|}{|GT(u)|}
\]

Enquanto Precision@K avalia a qualidade da lista recomendada, Recall@K mede a capacidade do sistema de recuperar os itens relevantes disponíveis.

In [42]:
def recall_at_k(recommended_items, relevant_items, k=10):
    if len(relevant_items) == 0:
        return 0.0

    recommended_k = recommended_items[:k]

    hits = len(set(recommended_k) & set(relevant_items))

    return hits / len(relevant_items)

In [43]:
recall_at_k(
    recommended_items=recommended,
    relevant_items=relevant,
    k=5
)

0.6666666666666666

### 5.3 NDCG@K

A Normalized Discounted Cumulative Gain (NDCG@K) avalia não apenas se itens relevantes foram recuperados, mas também em quais posições eles aparecem.

Itens relevantes posicionados no topo da lista recebem maior contribuição para a métrica.

Para relevância binária, o ganho acumulado descontado é definido como:

\[
DCG@K =
\sum_{j=1}^{K}
\frac{rel_j}{\log_2(j+1)}
\]

onde \(rel_j = 1\) quando o item na posição \(j\) é relevante e 0 caso contrário.

O valor é normalizado pelo melhor ranking possível:

\[
NDCG@K =
\frac{DCG@K}{IDCG@K}
\]

resultando em valores entre 0 e 1.

In [44]:
def ndcg_at_k(recommended_items, relevant_items, k=10):
    if len(relevant_items) == 0:
        return 0.0

    recommended_k = recommended_items[:k]

    dcg = 0.0

    for position, item_id in enumerate(recommended_k, start=1):
        if item_id in relevant_items:
            dcg += 1 / np.log2(position + 1)

    ideal_hits = min(len(relevant_items), k)

    idcg = sum(
        1 / np.log2(position + 1)
        for position in range(1, ideal_hits + 1)
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg

In [45]:
ndcg_at_k(
    recommended_items=recommended,
    relevant_items=relevant,
    k=5
)

np.float64(0.49818925746641285)

### 5.4 Validação com exemplos controlados

Antes de utilizar as funções em modelos reais, vale validar alguns casos simples.

In [46]:
#Caso 1 — ranking perfeito
recommended = [10, 20, 30]
relevant = {10, 20, 30}

print("Precision:", precision_at_k(recommended, relevant, k=3))
print("Recall:", recall_at_k(recommended, relevant, k=3))
print("NDCG:", ndcg_at_k(recommended, relevant, k=3))

Precision: 1.0
Recall: 1.0
NDCG: 1.0


In [47]:
#Caso 2 — nenhum acerto
recommended = [10, 20, 30]
relevant = {40, 50}

print("Precision:", precision_at_k(recommended, relevant, k=3))
print("Recall:", recall_at_k(recommended, relevant, k=3))
print("NDCG:", ndcg_at_k(recommended, relevant, k=3))

Precision: 0.0
Recall: 0.0
NDCG: 0.0


In [48]:
#Caso 3 — mesmos acertos, posições diferentes
relevant = {10, 20}

ranking_a = [10, 20, 30, 40]
ranking_b = [30, 40, 10, 20]

print(
    "NDCG ranking A:",
    ndcg_at_k(ranking_a, relevant, k=4)
)

print(
    "NDCG ranking B:",
    ndcg_at_k(ranking_b, relevant, k=4)
)

NDCG ranking A: 1.0
NDCG ranking B: 0.5706417189553201


## 6. Pipeline Comum de Avaliação

Para garantir uma comparação consistente entre os diferentes sistemas de recomendação, será utilizada uma função única de avaliação.

A função recebe as listas ordenadas de recomendações produzidas por um modelo e calcula Precision@K, Recall@K e NDCG@K para cada usuário elegível.

Os resultados individuais são posteriormente agregados pela média entre usuários.

In [50]:
def evaluate_recommendations(
    recommendations,
    ground_truth,
    k=10
):
    results = []

    for user_id, relevant_items in ground_truth.items():

        recommended_items = recommendations.get(
            user_id,
            []
        )

        precision = precision_at_k(
            recommended_items,
            relevant_items,
            k
        )

        recall = recall_at_k(
            recommended_items,
            relevant_items,
            k
        )

        ndcg = ndcg_at_k(
            recommended_items,
            relevant_items,
            k
        )

        results.append({
            "user_id": user_id,
            "precision": precision,
            "recall": recall,
            "ndcg": ndcg
        })

    results_df = pd.DataFrame(results)

    summary = {
        f"Precision@{k}": results_df["precision"].mean(),
        f"Recall@{k}": results_df["recall"].mean(),
        f"NDCG@{k}": results_df["ndcg"].mean(),
        "n_users": len(results_df)
    }

    return results_df, summary

## 7. Validação do Framework

Antes de utilizar o protocolo em modelos reais, será realizada uma validação integrada do pipeline de avaliação utilizando recomendações artificiais.

O objetivo é garantir que o ground truth, as listas Top-K e as métricas estejam sendo processados corretamente.

In [51]:
sample_ground_truth = {
    1: {10, 20},
    2: {30},
    3: {40, 50}
}

sample_recommendations = {
    1: [10, 30, 20, 40, 50],
    2: [10, 20, 30, 40, 50],
    3: [40, 60, 70, 80, 90]
}

sample_results, sample_summary = evaluate_recommendations(
    recommendations=sample_recommendations,
    ground_truth=sample_ground_truth,
    k=5
)

sample_results

,user_id,precision,recall,ndcg
0,1,0.4,1.0,0.919721
1,2,0.2,1.0,0.500000
2,3,0.2,0.5,0.613147


In [52]:
sample_summary

{'Precision@5': np.float64(0.26666666666666666),
 'Recall@5': np.float64(0.8333333333333334),
 'NDCG@5': np.float64(0.677622660637882),
 'n_users': 3}

## 8. Conclusões

O framework de avaliação foi construído para permitir a comparação consistente entre diferentes sistemas de recomendação.

O protocolo definido inclui:

- separação temporal das interações por usuário;
- definição explícita de relevância;
- construção do ground truth;
- exclusão de itens já observados;
- definição do universo de itens candidatos;
- tratamento de itens não disponíveis no catálogo de treino;
- métricas Precision@K, Recall@K e NDCG@K;
- uma função comum de avaliação para todos os modelos.

Com esse protocolo estabelecido, os próximos experimentos poderão concentrar-se exclusivamente na geração e ordenação das recomendações, mantendo constante a metodologia de avaliação.